In [ ]:
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 1215.5 gigabytes of available RAM

You are using a high-RAM runtime!


In [ ]:
!nvidia-smi

In [ ]:
# Import necessary libraries
import numpy as np
import onnxruntime as ort
import onnx
import os
import json
import ezkl
print(ezkl.__version__)
import torch
import torch.nn as nn
import torch.onnx
from torchvision.models import efficientnet_b0
from torchvision.models.efficientnet import SqueezeExcitation
from torch.utils.data import Subset, DataLoader, Dataset
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.utils import shuffle
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
import random
import albumentations as A
from albumentations.pytorch import ToTensorV2
import pandas as pd
from pathlib import Path
import tempfile
from onnx import helper, shape_inference, utils, TensorProto
import time, resource

In [ ]:
# Check for GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [ ]:
# Get detailed Tract/EZKL logs
os.environ["RUST_LOG"] = "trace"

# Set the visibility parameters
run_args = ezkl.PyRunArgs()
run_args.input_visibility = "private"
run_args.param_visibility = "private"
run_args.output_visibility = "public"
run_args.variables = [("batch_size", 1)]

#run_args.decomp_base = 65536
run_args.decomp_base = 1048576
#run_args.decomp_base = 16777216


In [ ]:
import os

# Define base and sub-paths
base_path = os.getcwd()
model_path = os.path.join(base_path, 'network.onnx')
input_data_path = os.path.join(base_path, 'input.json')
cal_data_path = os.path.join(base_path, 'cal_data.json')
calibration_data_path = os.path.join(base_path, 'calibration_data.json')

# Define folders
blocks_path = os.path.join(base_path, 'blocks')
calibration_path = os.path.join(base_path, 'calibration')
settings_path = os.path.join(base_path, 'settings')
compiled_model_path = os.path.join(base_path, 'compiled')
witness_path = os.path.join(base_path, 'witness')
proof_path = os.path.join(base_path, 'proof')

vk_path = os.path.join(base_path, 'vk')
pk_path = "/scratch/pioneer/jobs/pxk409/zk_setup/pk"
srs_path = os.path.join(base_path, 'aggregate/kzg_agg28.srs')

solidity_contract_path = os.path.join(base_path, 'verifier_contract.sol')
abi_path = os.path.join(base_path, 'verifier.abi')

# Create necessary directories if they don't exist
for path in [
    blocks_path,
    calibration_path,
    settings_path,
    compiled_model_path,
    witness_path,
    proof_path,
    vk_path,
    pk_path
]:
    os.makedirs(path, exist_ok=True)


In [ ]:
# Define directories of where the datasets reside
data_dir = '/home/pxk409/skin_lesions_13k'

# Define directories of where the training, validation and test sets reside
train_dir = os.path.join(data_dir, 'train')
test_dir = os.path.join(data_dir, 'test')

# Define directories of where the benign and malignant images reside
train_benign_dir = os.path.join(data_dir, 'train/benign')
train_malignant_dir = os.path.join(data_dir, 'train/malignant')
test_benign_dir = os.path.join(data_dir, 'test/benign')
test_malignant_dir = os.path.join(data_dir, 'test/malignant')

# Load full dataset
full_dataset = datasets.ImageFolder(root=train_dir)

# Extract labels from the dataset
labels = [label for _, label in full_dataset.samples]

# Stratified split (80% train, 20% validation)
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=12345)
train_idx, val_idx = next(split.split(np.zeros(len(labels)), labels))

# Create training and validation subsets
train_dataset = Subset(full_dataset, train_idx)
val_dataset = Subset(full_dataset, val_idx)

# Custom transform to integrate Albumentations with PyTorch
class AlbumentationsTransform:
    def __init__(self, augmentations):
        self.augmentations = A.Compose(augmentations)

    def __call__(self, image):
        image = np.array(image)  # Convert PIL Image to NumPy array
        augmented = self.augmentations(image=image)
        return augmented["image"]

# Simple augmentations for training
simple_transform = AlbumentationsTransform([
    A.RandomResizedCrop(height=224, width=224, scale=(0.8, 1.0), p=1.0),  # Random crop and resize
    A.HorizontalFlip(p=0.5),  # Horizontal flip
    A.VerticalFlip(p=0.5),  # Vertical flip
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Normalize to match pre-trained model
    ToTensorV2(),  # Convert to PyTorch tensor and ensure dtype=torch.float32
])

# Use AlbumentationsTransform directly in the loaders for consistency
train_transform = simple_transform

# Albumentations augmentations for validation and test
albumentations_val_test_transform = AlbumentationsTransform([
    A.Resize(height=224, width=224),  # Resize to EfficientNet input size
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Normalize to match pre-trained model
    ToTensorV2()  # Convert to PyTorch tensor
])

test_val_transform = albumentations_val_test_transform

class TransformDataset(Dataset):
    def __init__(self, dataset, transform):
        """
        Args:
            dataset: Original dataset (e.g., Subset of ImageFolder).
            transform: Transform to be applied to the images in the dataset.
        """
        self.dataset = dataset
        self.transform = transform

    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

    def __len__(self):
        return len(self.dataset)

# Wrap subsets with the respective transforms
train_dataset = TransformDataset(train_dataset, train_transform)
val_dataset = TransformDataset(val_dataset, test_val_transform)

# Create data loaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# Channel Attention
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction_ratio):
        super(ChannelAttention, self).__init__()
        # Global descriptors from avg and max pooling
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        # Shared MLP for attention generation
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction_ratio, bias=False),
            nn.ReLU(),
            nn.Linear(in_channels // reduction_ratio, in_channels, bias=False)
        )

        # Activation to produce the attention map
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.size()

        # Generate global descriptors
        avg_out = self.fc(self.avg_pool(x).view(b, c))  # Avg pooling path
        max_out = self.fc(self.max_pool(x).view(b, c))  # Max pooling path

        # Combine and produce channel attention
        out = avg_out + max_out
        return self.sigmoid(out).view(b, c, 1, 1)  # Reshape for channel-wise multiplication


# Global Context Block
class GlobalContextBlock(nn.Module):
    def __init__(self, in_channels, reduction_ratio):
        super(GlobalContextBlock, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction_ratio, bias=False),
            nn.ReLU(),
            nn.Linear(in_channels // reduction_ratio, in_channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, h, w = x.size()
        # Compute global context vector
        context = x.mean(dim=(2, 3), keepdim=False)  # Global average pooling
        context = self.fc(context)  # Apply FC layers
        return x * context.view(b, c, 1, 1)  # Broadcast context to match spatial dimensions


# Multi-Scale Spatial Attention
class MultiScaleSpatialAttention(nn.Module):
    def __init__(self, kernel_sizes=[3, 5, 7, 9]):
        super(MultiScaleSpatialAttention, self).__init__()
        self.spatial_convs = nn.ModuleList([
            nn.Conv2d(2, 1, kernel_size=ks, padding=(ks - 1) // 2, bias=False) # Same padding
            for ks in kernel_sizes
        ])
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)  # Avg pooling path
        max_out, _ = torch.max(x, dim=1, keepdim=True)  # Max pooling path
        combined = torch.cat([avg_out, max_out], dim=1)  # Combine features

        # Apply each spatial attention head
        multi_scale_outputs = [self.sigmoid(conv(combined)) for conv in self.spatial_convs]
        # Combine multi-scale outputs
        out = torch.sum(torch.stack(multi_scale_outputs, dim=0), dim=0)
        return out * x  # Apply spatial attention


# Spatial Dropout Class
class SpatialDropout(nn.Module):
    def __init__(self, p=0):
        super(SpatialDropout, self).__init__()
        self.p = p

    def forward(self, x):
        # Apply channel-wise dropout
        return nn.functional.dropout2d(x, self.p, training=self.training)



# HCAM Block
class HCAM(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16, kernel_sizes=[3, 5, 7, 9]):
        super(HCAM, self).__init__()
        self.channel_attention = ChannelAttention(in_channels, reduction_ratio)
        self.global_context_block = GlobalContextBlock(in_channels, reduction_ratio)
        self.spatial_attention = MultiScaleSpatialAttention(kernel_sizes)

    def forward(self, x):
        # Apply channel attention
        out = self.channel_attention(x) * x
        # Apply global context block
        out = self.global_context_block(out)
        # Apply multi-scale spatial attention
        out = self.spatial_attention(out)
        return out


# Replace SE blocks with HCAM and add spatial dropout
def replace_se_with_hcam(module, dropout_rate=0):
    for name, layer in module._modules.items():
        if isinstance(layer, SqueezeExcitation):
            in_channels = layer.fc1.in_channels
            # Replace SE block with HCAM and add Spatial Dropout
            module._modules[name] = nn.Sequential(
                HCAM(in_channels=in_channels)#,
                #SpatialDropout(p=dropout_rate)
            )
            #print(f"Replaced SE block with Enhanced CBAM and added Spatial Dropout in: {name}")
        elif isinstance(layer, nn.Sequential) or isinstance(layer, nn.Module):
            replace_se_with_hcam(layer, dropout_rate)

In [ ]:
def initialize_model(K=2):
    # Load pretrained EfficientNet-B0
    model = efficientnet_b0(weights="DEFAULT")

    # Select only the last K layers of model.features
    last_k_layers = list(model.features.children())[-K:]

    # Replace SE blocks with Enhanced CBAM in the last K layers
    replace_se_with_hcam(nn.Sequential(*last_k_layers))

    # Freeze weights for the remaining layers (first len(features) - K layers)
    for layer in list(model.features.children())[:-K]:
        for param in layer.parameters():
            param.requires_grad = False

    # Modify the classifier for binary classification
    model.classifier = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(model.classifier[1].in_features, 512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, 2)
    )

    # Move model to device
    #model = model.to(device)
    return model

# Initialize the model
model = initialize_model()

# Load the trained weights
model.load_state_dict(torch.load("best_model.pth", weights_only=True)) #, map_location=torch.device('cpu')
model.to(device)

In [ ]:
# Retrieve the first batch from the validation data loader
first_batch_images, first_batch_labels = next(iter(val_loader))

# Select the first and second samples from the batch
first_sample_image = first_batch_images[0]
first_sample_label = first_batch_labels[0]

second_sample_image = first_batch_images[1]
second_sample_label = first_batch_labels[1]

# Print shapes for debugging
print(f"First batch images shape: {first_batch_images.shape}")
print(f"First batch labels shape: {first_batch_labels.shape}")
print(f"First sample image shape: {first_sample_image.shape}")
print(f"First sample label: {first_sample_label}")
print(f"Second sample image shape: {second_sample_image.shape}")
print(f"Second sample label: {second_sample_label}")

# Create a batch of size one for the first and second samples
first_image_batch = first_sample_image.unsqueeze(0)
second_image_batch = second_sample_image.unsqueeze(0)

first_label_batch = torch.tensor([first_sample_label])
second_label_batch = torch.tensor([second_sample_label])

# Print shapes for debugging
print(f"First image batch shape: {first_image_batch.shape}")
print(f"First label batch shape: {first_label_batch.shape}")

First batch images shape: torch.Size([64, 3, 224, 224])
First batch labels shape: torch.Size([64])
First sample image shape: torch.Size([3, 224, 224])
First sample label: 1
Second sample image shape: torch.Size([3, 224, 224])
Second sample label: 1
First image batch shape: torch.Size([1, 3, 224, 224])
First label batch shape: torch.Size([1])


In [ ]:
# Create a dummy input with the determined size
dummy_input = torch.randn(1, 3, 224, 224, device=device)


# Set the model to evaluation mode
model.eval()

# Move the input tensor (first_image_batch) to the GPU
first_image_batch = first_image_batch.to(device)

# Export the model to ONNX format
torch.onnx.export(
    model,
    dummy_input,
    model_path,               # Where to save the model
    export_params=True,       # Store the trained parameter weights inside the model
    opset_version=11,         # ONNX version
    do_constant_folding=True, # Optimize the model
    input_names=['input'],    # Name the model input
    output_names=['output'],  # Name the model output
    dynamic_axes={
        'input': {0: 'batch_size'},  # Allow batch size to be dynamic
        'output': {0: 'batch_size'}
    }
)

print(f"Model has been exported to {model_path}.")

Model has been exported to /home/pxk409/Skin_EZKL/network.onnx.


In [ ]:
from onnx import load_model

def list_onnx_nodes(model_path):
    model = load_model(model_path)
    for i, node in enumerate(model.graph.node):
        print(f"Node {i+1}: Name={node.name}, Inputs={node.input}, Outputs={node.output}")

# List node names
list_onnx_nodes(model_path)


Node 1: Name=Identity_8, Inputs=['onnx::MatMul_876'], Outputs=['onnx::MatMul_878']
Node 2: Name=Identity_9, Inputs=['onnx::MatMul_875'], Outputs=['onnx::MatMul_877']
Node 3: Name=/features/features.0/features.0.0/Conv, Inputs=['input', 'onnx::Conv_729', 'onnx::Conv_730'], Outputs=['/features/features.0/features.0.0/Conv_output_0']
Node 4: Name=/features/features.0/features.0.2/Sigmoid, Inputs=['/features/features.0/features.0.0/Conv_output_0'], Outputs=['/features/features.0/features.0.2/Sigmoid_output_0']
Node 5: Name=/features/features.0/features.0.2/Mul, Inputs=['/features/features.0/features.0.0/Conv_output_0', '/features/features.0/features.0.2/Sigmoid_output_0'], Outputs=['/features/features.0/features.0.2/Mul_output_0']
Node 6: Name=/features/features.1/features.1.0/block/block.0/block.0.0/Conv, Inputs=['/features/features.0/features.0.2/Mul_output_0', 'onnx::Conv_732', 'onnx::Conv_733'], Outputs=['/features/features.1/features.1.0/block/block.0/block.0.0/Conv_output_0']
Node 7:

In [ ]:
#new -- now old

block_definitions = [

    # ───────── Block 1  (nodes 3 → 5) ─────────
    ("input",
     "/features/features.0/features.0.2/Mul_output_0"),                      # node 5

    # ───────── Block 2  (nodes 6 → 16) ────────
    ("/features/features.0/features.0.2/Mul_output_0",                       # node 5
     "/features/features.1/features.1.0/block/block.2/"
     "block.2.0/Conv_output_0"),                                             # node 16

    # ───────── Block 3  (nodes 17) ────────────
    ("/features/features.1/features.1.0/block/block.2/"
     "block.2.0/Conv_output_0",                                              # node 16
     "/features/features.2/features.2.0/block/block.0/"
     "block.0.0/Conv_output_0"),                                             # node 17

    # ───────── Block 4  (nodes 18 → 22) ──────
    ("/features/features.2/features.2.0/block/block.0/"
     "block.0.0/Conv_output_0",                                              # node 17
     "/features/features.2/features.2.0/block/block.1/"
     "block.1.2/Mul_output_0"),                                              # node 22

    # ───────── Block 5  (nodes 23 → 30) ──────
    ("/features/features.2/features.2.0/block/block.1/"
     "block.1.2/Mul_output_0",                                               # node 22
     "/features/features.2/features.2.0/block/block.3/"
     "block.3.0/Conv_output_0"),                                             # node 30

    # ───────── Block 6  (nodes 31 → 45) ──────
    ("/features/features.2/features.2.0/block/block.3/"
     "block.3.0/Conv_output_0",                                              # node 30
     "/features/features.2/features.2.1/Add_output_0"),                      # node 45

    # ───────── Block 7  (nodes 46 → 58) ──────
    ("/features/features.2/features.2.1/Add_output_0",                       # node 45
     "/features/features.3/features.3.0/block/block.2/Mul_output_0"),        # node 58

    # ───────── Block 8  (nodes 59 → 74) ──────
    ("/features/features.3/features.3.0/block/block.2/Mul_output_0",         # node 58
     "/features/features.3/features.3.1/Add_output_0"),                      # node 74

    # ───────── Block 9  (nodes 75 → 103) ─────
    ("/features/features.3/features.3.1/Add_output_0",                       # node 74
     "/features/features.4/features.4.1/Add_output_0"),                      # node 103

    # ───────── Block 10 (nodes 104 → 131) ────
    ("/features/features.4/features.4.1/Add_output_0",                       # node 103
     "/features/features.5/features.5.0/block/block.2/Mul_output_0"),        # node 131

    # ───────── Block 11 (nodes 132 → 147) ────
    ("/features/features.5/features.5.0/block/block.2/Mul_output_0",         # node 131
     "/features/features.5/features.5.1/Add_output_0"),                      # node 147

    # ───────── Block 12 (nodes 148 → 162) ────
    ("/features/features.5/features.5.1/Add_output_0",                       # node 147
     "/features/features.5/features.5.2/Add_output_0"),                      # node 162

    # ───────── Block 13 (nodes 163 → 191) ────
    ("/features/features.5/features.5.2/Add_output_0",                       # node 162
     "/features/features.6/features.6.1/Add_output_0"),                      # node 191

    # ───────── Block 14 (nodes 192 → 206) ────
    ("/features/features.6/features.6.1/Add_output_0",                       # node 191
     "/features/features.6/features.6.2/Add_output_0"),                      # node 206

    # ───────── Block 15 (nodes 207 → 221) ────
    ("/features/features.6/features.6.2/Add_output_0",                       # node 206
     "/features/features.6/features.6.3/Add_output_0"),                      # node 221

    # ───────── Block 16 (nodes 222 → 305) ────
    ("/features/features.6/features.6.3/Add_output_0",                       # node 221
     "output"),                                                              # node 305
]


from onnx import utils

def split_model_by_blocks(model_path, block_definitions, output_base_path):
    """
    Splits an ONNX model into blocks based on predefined input and output nodes.

    Args:
    - model_path (str): Path to the ONNX model.
    - block_definitions (list of tuples): Each tuple contains (start_node, end_node).
    - output_base_path (str): Base path to save the split models (e.g., 'block').

    Returns:
    - None
    """
    for i, (start, end) in enumerate(block_definitions):
        block_path = f"{output_base_path}/blocks/block_{i+1}.onnx"
        try:
            # Extract and save the sub-model directly
            utils.extract_model(
                model_path,
                input_names=[start],
                output_names=[end],
                output_path=block_path
            )
            print(f"Saved Block {i+1} to {block_path}")
        except Exception as e:
            #print(f"Failed to split Block {i+1}, Error: {e}")
            print(f"Failed to split Block {i+1}, Start={start}, End={end}, Error: {e}")

# Split the model
split_model_by_blocks(model_path, block_definitions, base_path)


Saved Block 1 to /home/pxk409/Skin_EZKL/blocks/block_1.onnx
Saved Block 2 to /home/pxk409/Skin_EZKL/blocks/block_2.onnx
Saved Block 3 to /home/pxk409/Skin_EZKL/blocks/block_3.onnx
Saved Block 4 to /home/pxk409/Skin_EZKL/blocks/block_4.onnx
Saved Block 5 to /home/pxk409/Skin_EZKL/blocks/block_5.onnx
Saved Block 6 to /home/pxk409/Skin_EZKL/blocks/block_6.onnx
Saved Block 7 to /home/pxk409/Skin_EZKL/blocks/block_7.onnx
Saved Block 8 to /home/pxk409/Skin_EZKL/blocks/block_8.onnx
Saved Block 9 to /home/pxk409/Skin_EZKL/blocks/block_9.onnx
Saved Block 10 to /home/pxk409/Skin_EZKL/blocks/block_10.onnx
Saved Block 11 to /home/pxk409/Skin_EZKL/blocks/block_11.onnx
Saved Block 12 to /home/pxk409/Skin_EZKL/blocks/block_12.onnx
Saved Block 13 to /home/pxk409/Skin_EZKL/blocks/block_13.onnx
Saved Block 14 to /home/pxk409/Skin_EZKL/blocks/block_14.onnx
Saved Block 15 to /home/pxk409/Skin_EZKL/blocks/block_15.onnx
Saved Block 16 to /home/pxk409/Skin_EZKL/blocks/block_16.onnx


In [ ]:
# Serialize the first sample to create the input data file
x = first_image_batch
data_array = x.detach().cpu().numpy().reshape(-1).tolist()

data = dict(input_data=[data_array])

with open(input_data_path, 'w') as input_file:
    json.dump(data, input_file)

print(f"Input data saved to {input_data_path}.")

# Serialize the second sample to create the calibration data file
cal_data_array = second_image_batch.detach().cpu().numpy().reshape(-1).tolist()

cal_data = dict(input_data=[cal_data_array])

with open(cal_data_path, 'w') as cal_file:
    json.dump(cal_data, cal_file)

print(f"Calibration data saved to {cal_data_path}.")

Input data saved to /home/pxk409/Skin_EZKL/input.json.
Calibration data saved to /home/pxk409/Skin_EZKL/cal_data.json.


In [ ]:
# Retrieve a batch from the validation data loader
batch_images, batch_labels = next(iter(val_loader))

# Print shapes for debugging
print(f"Batch images shape: {batch_images.shape}")
print(f"Batch labels shape: {batch_labels.shape}")

# Serialize the entire batch as calibration data
batch_array = batch_images.detach().cpu().numpy().reshape(batch_images.size(0), -1).tolist()

# Prepare the calibration data dictionary
cal_data = dict(input_data=batch_array)

# Save the calibration data to a JSON file
with open(calibration_data_path, 'w') as cal_file:
    json.dump(cal_data, cal_file)

print(f"Calibration data saved to {calibration_data_path}.")

Batch images shape: torch.Size([64, 3, 224, 224])
Batch labels shape: torch.Size([64])
Calibration data saved to /home/pxk409/Skin_EZKL/calibration_data.json.


In [ ]:
import ezkl, pprint, inspect

run_args = ezkl.PyRunArgs()
pprint.pprint([a for a in dir(run_args) if not a.startswith("_")])

# And list every enum exported by the module
for name, obj in vars(ezkl).items():
    if inspect.isclass(obj) and "enum" in str(type(obj)).lower():
        print(name, list(obj))


['bounded_log_lookup',
 'check_mode',
 'commitment',
 'decomp_base',
 'decomp_legs',
 'epsilon',
 'ignore_range_check_inputs_outputs',
 'input_scale',
 'input_visibility',
 'logrows',
 'lookup_range',
 'num_inner_cols',
 'output_visibility',
 'param_scale',
 'param_visibility',
 'rebase_frac_zero_constants',
 'scale_rebase_multiplier',
 'variables']


In [ ]:
agg29_srs_path = os.path.join(base_path, "aggregate", "kzg_agg29.srs")

ezkl.get_srs(
    settings_path=None,
    logrows=29,
    srs_path=agg29_srs_path,
    commitment=ezkl.PyCommitments.KZG
)

<Future pending cb=[<builtins.PyDoneCallback object at 0x7f661c27bdb0>()]>

In [ ]:
# Generate Settings for Each Block

def generate_settings(blocks_path, output_settings_path):
    """
    Generates EZKL settings for each ONNX block.

    Args:
    - blocks_path (str): Directory where the blocks are saved.
    - output_settings_path (str): Directory to save the settings.

    Returns:
    - None
    """
    for i in range(1, 17):
        block_path = f"{blocks_path}/block_{i}.onnx"
        settings_path = f"{output_settings_path}/block_{i}_settings.json"
        try:
            ezkl.gen_settings(
                model=block_path,
                output=settings_path,
                py_run_args=run_args
            )
            print(f"Generated settings for Block {i} at {settings_path}")
        except Exception as e:
            print(f"Failed to generate settings for Block {i}, Error: {e}")

# Call the function
generate_settings(blocks_path, settings_path)

Generated settings for Block 1 at /home/pxk409/Skin_EZKL/settings/block_1_settings.json
Generated settings for Block 2 at /home/pxk409/Skin_EZKL/settings/block_2_settings.json
Generated settings for Block 3 at /home/pxk409/Skin_EZKL/settings/block_3_settings.json
Generated settings for Block 4 at /home/pxk409/Skin_EZKL/settings/block_4_settings.json
Generated settings for Block 5 at /home/pxk409/Skin_EZKL/settings/block_5_settings.json
Generated settings for Block 6 at /home/pxk409/Skin_EZKL/settings/block_6_settings.json
Generated settings for Block 7 at /home/pxk409/Skin_EZKL/settings/block_7_settings.json
Generated settings for Block 8 at /home/pxk409/Skin_EZKL/settings/block_8_settings.json
Generated settings for Block 9 at /home/pxk409/Skin_EZKL/settings/block_9_settings.json
Generated settings for Block 10 at /home/pxk409/Skin_EZKL/settings/block_10_settings.json
Generated settings for Block 11 at /home/pxk409/Skin_EZKL/settings/block_11_settings.json
Generated settings for Block

In [ ]:
import glob, json, os, pprint

def list_r1cs_rows(settings_dir="/path/to/settings"):
    """
    Scan `settings_dir` for files named like: block_<index>_settings.json,
    print the `num_rows` field for each, and also report the grand total.
    """
    pattern = os.path.join(settings_dir, "block_*_settings*.json")
    rows_per_block = {}
    total_rows = 0  # NEW: running sum

    for fp in glob.glob(pattern):
        # Derive the block index from the filename
        block_part = os.path.basename(fp).split('_')[1]
        block_idx  = int(''.join(filter(str.isdigit, block_part)))

        with open(fp, "r") as f:
            data = json.load(f)

        num_rows = data.get("num_rows", 0)
        rows_per_block[block_idx] = num_rows
        total_rows += num_rows

    # Pretty-print in ascending block order
    for blk in sorted(rows_per_block):
        print(f"Block {blk:2d}: {rows_per_block[blk]:,} rows")

    # NEW: print and return the total
    print("-" * 40)
    print(f"   TOTAL: {total_rows:,} rows")

    return rows_per_block, total_rows

# ------------------------------------------------------------------
_, grand_total = list_r1cs_rows("/home/pxk409/Skin_EZKL/settings")


Block  1: 37,606,985 rows
Block  2: 53,992,077 rows
Block  3: 66,633,822 rows
Block  4: 70,898,773 rows
Block  5: 18,521,337 rows
Block  6: 84,945,367 rows
Block  7: 46,617,127 rows
Block  8: 43,758,308 rows
Block  9: 45,011,334 rows
Block 10: 43,434,038 rows
Block 11: 43,970,915 rows
Block 12: 37,477,793 rows
Block 13: 42,949,179 rows
Block 14: 20,679,142 rows
Block 15: 20,679,133 rows
Block 16: 43,367,074 rows
----------------------------------------
   TOTAL: 720,542,404 rows


In [ ]:
def generate_calibration_data_for_blocks(blocks_path, initial_calibration_file, output_calibration_path):
    """
    Generates calibration data for all ONNX blocks by propagating the input data through the blocks.

    Args:
    - blocks_path (str): Directory where the ONNX blocks are saved.
    - initial_calibration_file (str): Path to the calibration data file for the first block.
    - output_calibration_path (str): Directory to save the calibration data for each block.

    Returns:
    - None
    """
    # Copy the initial calibration data as block_1_calibration.json
    block_1_cal_data_path = os.path.join(output_calibration_path, "block_1_calibration.json")
    with open(initial_calibration_file, 'r') as cal_file:
        calibration_data = json.load(cal_file)
    with open(block_1_cal_data_path, 'w') as cal_file:
        json.dump(calibration_data, cal_file)
    print(f"Saved initial calibration data as {block_1_cal_data_path}")

    # Start generating calibration data for subsequent blocks
    current_input_data = np.array(calibration_data["input_data"][0], dtype=np.float32)

    # Load the initial calibration data for the first block
    #with open(initial_calibration_file, 'r') as cal_file:
    #    calibration_data = json.load(cal_file)
    #    current_input_data = np.array(calibration_data["input_data"][0], dtype=np.float32)

    for i in range(1, 16):  # Loop through blocks (no need to use the last block)
        block_path = os.path.join(blocks_path, f"block_{i}.onnx")
        cal_data_file = os.path.join(output_calibration_path, f"block_{i+1}_calibration.json")

        try:
            print(f"Generating calibration data for Block {i+1}...")

            # Load ONNX block
            session_options = ort.SessionOptions()
            session_options.intra_op_num_threads = 1  # Avoid thread affinity issues
            session_options.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL

            ort_session = ort.InferenceSession(block_path, sess_options=session_options)
            input_name = ort_session.get_inputs()[0].name
            input_shape = ort_session.get_inputs()[0].shape

            # Replace dynamic dimensions (e.g., 'batch_size') with a value (e.g., 1)
            input_shape = [1 if dim is None or isinstance(dim, str) else dim for dim in input_shape]

            # Ensure current input data size matches expected size
            expected_size = np.prod(input_shape)
            if current_input_data.size != expected_size:
                raise ValueError(f"Block {i+1}: Expected {expected_size} values, got {current_input_data.size}.")

            # Reshape current input data to match the block's input shape
            reshaped_input = current_input_data.reshape(input_shape)

            # Run inference and capture output
            outputs = ort_session.run(None, {input_name: reshaped_input})
            current_input_data = outputs[0]  # Update input for the next block

            # Flatten and save as calibration data
            flattened_output = current_input_data.flatten().tolist()
            cal_data = {"input_data": [flattened_output]}

            with open(cal_data_file, 'w') as cal_file:
                json.dump(cal_data, cal_file)

            print(f"Calibration data for Block {i+1} saved to {cal_data_file}")

        except Exception as e:
            print(f"Failed to generate calibration data for Block {i+1}, Error: {e}")

# Run the calibration generation
generate_calibration_data_for_blocks(
    blocks_path=blocks_path,
    initial_calibration_file=cal_data_path,
    output_calibration_path=calibration_path
)

Saved initial calibration data as /home/pxk409/Skin_EZKL/calibration/block_1_calibration.json
Generating calibration data for Block 2...
Calibration data for Block 2 saved to /home/pxk409/Skin_EZKL/calibration/block_2_calibration.json
Generating calibration data for Block 3...
Calibration data for Block 3 saved to /home/pxk409/Skin_EZKL/calibration/block_3_calibration.json
Generating calibration data for Block 4...
Calibration data for Block 4 saved to /home/pxk409/Skin_EZKL/calibration/block_4_calibration.json
Generating calibration data for Block 5...
Calibration data for Block 5 saved to /home/pxk409/Skin_EZKL/calibration/block_5_calibration.json
Generating calibration data for Block 6...
Calibration data for Block 6 saved to /home/pxk409/Skin_EZKL/calibration/block_6_calibration.json
Generating calibration data for Block 7...
Calibration data for Block 7 saved to /home/pxk409/Skin_EZKL/calibration/block_7_calibration.json
Generating calibration data for Block 8...
Calibration data 

In [ ]:
# Calibrate Settings for Each Block

import asyncio
async def calibrate_block_settings(blocks_path, calibration_path, settings_path):
    """
    Runs `ezkl.calibrate_settings` for each ONNX block using its calibration data.

    Args:
    - blocks_path (str): Path to the directory containing ONNX blocks.
    - calibration_path (str): Path to the directory containing calibration data for each block.
    - settings_path (str): Directory to save the calibrated settings files.

    Returns:
    - None
    """
    for i in range(1, 17):  # Assuming 16 blocks
        block_path = os.path.join(blocks_path, f"block_{i}.onnx")
        cal_data_file = os.path.join(calibration_path, f"block_{i}_calibration.json")
        settings_file = os.path.join(settings_path, f"block_{i}_settings.json")

        try:
            print(f"Calibrating settings for Block {i}...")

            # Call ezkl.calibrate_settings directly
            await ezkl.calibrate_settings(
                    data                     = cal_data_file,
                    model                    = block_path,
                    settings                 = settings_file,
                    target                   = "resources",
                    #max_logrows              = 24,
                    lookup_safety_margin     = 1,
                    scales                   = [0, 4],
                    scale_rebase_multiplier  = [4]
            )

            print(f"Calibrated settings for Block {i} saved to {settings_file}")

        except Exception as e:
            print(f"Failed to calibrate settings for Block {i}, Error: {e}")


# Calibrate settings for blocks
await calibrate_block_settings(
    blocks_path=blocks_path,
    calibration_path=calibration_path,
    settings_path=settings_path
)


Calibrating settings for Block 1...




 <------------- Numerical Fidelity Report (input_scale: 4, param_scale: 4, scale_input_multiplier: 4) ------------->

+--------------+--------------+-----------+-----------+----------------+------------------+---------------+---------------+--------------------+--------------------+------------------------+
| mean_error   | median_error | max_error | min_error | mean_abs_error | median_abs_error | max_abs_error | min_abs_error | mean_squared_error | mean_percent_error | mean_abs_percent_error |
+--------------+--------------+-----------+-----------+----------------+------------------+---------------+---------------+--------------------+--------------------+------------------------+
| 0.0045066443 | 0.03642583   | 1.0322394 | -1.170601 | 0.05483422     | 0.03642583       | 1.170601      | 0             | 0.008171307        | 0.11550888         | 0.3229177              |
+--------------+--------------+-----------+-----------+----------------+------------------+---------------+---------

Calibrated settings for Block 1 saved to /home/pxk409/Skin_EZKL/settings/block_1_settings.json
Calibrating settings for Block 2...




 <------------- Numerical Fidelity Report (input_scale: 0, param_scale: 0, scale_input_multiplier: 4) ------------->

+------------+--------------+-----------+------------+----------------+------------------+---------------+----------------+--------------------+--------------------+------------------------+
| mean_error | median_error | max_error | min_error  | mean_abs_error | median_abs_error | max_abs_error | min_abs_error  | mean_squared_error | mean_percent_error | mean_abs_percent_error |
+------------+--------------+-----------+------------+----------------+------------------+---------------+----------------+--------------------+--------------------+------------------------+
| -3.8602693 | 9.727434     | 44.006374 | -31.591833 | 10.826081      | 9.727434         | 44.006374     | 0.000005722046 | 179.58444          | 30.943472          | 87.81406               |
+------------+--------------+-----------+------------+----------------+------------------+---------------+----------

Calibrated settings for Block 2 saved to /home/pxk409/Skin_EZKL/settings/block_2_settings.json
Calibrating settings for Block 3...


...

Calibrated settings for Block 14 saved to /home/pxk409/Skin_EZKL/settings/block_14_settings.json
Calibrating settings for Block 15...




 <------------- Numerical Fidelity Report (input_scale: 0, param_scale: 4, scale_input_multiplier: 4) ------------->

+------------+--------------+-----------+-----------+----------------+------------------+---------------+---------------+--------------------+--------------------+------------------------+
| mean_error | median_error | max_error | min_error | mean_abs_error | median_abs_error | max_abs_error | min_abs_error | mean_squared_error | mean_percent_error | mean_abs_percent_error |
+------------+--------------+-----------+-----------+----------------+------------------+---------------+---------------+--------------------+--------------------+------------------------+
| 0.09499482 | -0.06801236  | 13.562447 | -9.700467 | 1.6738455      | 0.06801236       | 13.562447     | 0.001124382   | 5.263656           | 0.24785358         | 2.262146               |
+------------+--------------+-----------+-----------+----------------+------------------+---------------+---------------+---

Calibrated settings for Block 15 saved to /home/pxk409/Skin_EZKL/settings/block_15_settings.json
Calibrating settings for Block 16...




 <------------- Numerical Fidelity Report (input_scale: 4, param_scale: 4, scale_input_multiplier: 4) ------------->

+-------------+--------------+-------------+---------------+----------------+------------------+---------------+---------------+--------------------+--------------------+------------------------+
| mean_error  | median_error | max_error   | min_error     | mean_abs_error | median_abs_error | max_abs_error | min_abs_error | mean_squared_error | mean_percent_error | mean_abs_percent_error |
+-------------+--------------+-------------+---------------+----------------+------------------+---------------+---------------+--------------------+--------------------+------------------------+
| 0.015695132 | 0.033557363  | 0.033557363 | -0.0021670982 | 0.01786223     | 0.033557363      | 0.033557363   | 0.0021670982  | 0.00056539645      | 1.7360963          | 1.7927247              |
+-------------+--------------+-------------+---------------+----------------+------------------+

Calibrated settings for Block 16 saved to /home/pxk409/Skin_EZKL/settings/block_16_settings.json


In [ ]:
import os
import json

settings_dir = '/home/pxk409/Skin_EZKL/settings'

# List all JSON files in the settings directory
json_files = [f for f in os.listdir(settings_dir) if f.endswith('.json')]

for fname in json_files:
    fpath = os.path.join(settings_dir, fname)
    try:
        with open(fpath, 'r') as f:
            cfg = json.load(f)
    except (json.JSONDecodeError, OSError) as e:
        print(f"File: {fname} — error reading JSON: {e}")
        continue

    # logrows is nested under run_args in EZKL‑generated setting files
    logrows = cfg.get('run_args', {}).get('logrows')
    if logrows is None:
        print(f"File: {fname} — 'logrows' not found in run_args")
    else:
        print(f"File: {fname}, logrows: {logrows}")


File: block_10_settings.json, logrows: 24
File: block_11_settings.json, logrows: 25
File: block_12_settings.json, logrows: 25
File: block_13_settings.json, logrows: 25
File: block_14_settings.json, logrows: 24
File: block_15_settings.json, logrows: 24
File: block_16_settings.json, logrows: 25
File: block_1_settings.json, logrows: 24
File: block_2_settings.json, logrows: 24
File: block_3_settings.json, logrows: 25
File: block_4_settings.json, logrows: 25
File: block_5_settings.json, logrows: 23
File: block_6_settings.json, logrows: 25
File: block_7_settings.json, logrows: 24
File: block_8_settings.json, logrows: 24
File: block_9_settings.json, logrows: 24


In [ ]:
import os
import json
import math

# ----------- paths & params -----------------
base_path       = "/home/pxk409/Skin_EZKL"
proof_path      = os.path.join(base_path, "proof")
aggregate_path  = os.path.join(base_path, "aggregate")

os.makedirs(aggregate_path, exist_ok=True)

# Automatically detect all proof files
proof_files = sorted([
    os.path.join(proof_path, f)
    for f in os.listdir(proof_path)
    if f.startswith("block_") and f.endswith("_proof.json")
])

# Output aggregation file paths
agg_srs_path    = os.path.join(aggregate_path, "agg_kzg.srs")
agg_pk_path     = os.path.join(aggregate_path, "agg.pk")
agg_vk_path     = os.path.join(aggregate_path, "agg.vk")
agg_proof_path  = os.path.join(aggregate_path, "agg_proof.json")

# ----------- extract and print inner logrows (k) -----------------
inner_ks = []
for p in proof_files:
    try:
        with open(p) as f:
            meta = json.load(f)
        k = meta.get("protocol", {}).get("domain", {}).get("k")
        if k is None:
            raise ValueError(f"'k' not found in {p}")
        inner_ks.append(k)
        print(f"🔹 {os.path.basename(p)} → k = {k}")
    except Exception as e:
        print(f"❌ Error reading {p}: {e}")

# ----------- compute aggregation logrows -----------------
if not inner_ks:
    raise RuntimeError("No valid proof files found to compute aggregation logrows.")

num_proofs  = len(inner_ks)
agg_logrows = max(inner_ks) + math.ceil(math.log2(num_proofs)) + 1

print(f"\n✅ Computed aggregation logrows = {agg_logrows}")
print(f"📂 Proofs aggregated: {num_proofs}")

In [ ]:
%%time
!RUST_LOG=trace

# Generate the settings file
print("Generating ezkl settings...")
res = ezkl.gen_settings(model=model_path, output=settings_path, py_run_args=run_args)
assert res is True, "Failed to generate settings"
print("done")

Generating ezkl settings...
done
CPU times: user 16min 15s, sys: 3min 24s, total: 19min 40s
Wall time: 5min 15s


In [ ]:
# Compile Circuit

def compile_all_circuits(blocks_path, settings_path, compiled_model_path):
    """
    Compiles all ONNX blocks into circuits.

    Args:
    - blocks_path (str): Path to the directory containing ONNX blocks.
    - settings_path (str): Path to the directory containing settings files.
    - compiled_model_path (str): Path to the directory where compiled circuits will be saved.

    Returns:
    - None
    """
    for i in range(1, 17):
        block_path = os.path.join(blocks_path, f"block_{i}.onnx")
        settings_file = os.path.join(settings_path, f"block_{i}_settings.json")
        circuit_file = os.path.join(compiled_model_path, f"block_{i}.compiled")

        try:
            print(f"Compiling circuit for Block {i}...")
            ezkl.compile_circuit(
                model=block_path,
                settings_path=settings_file,
                compiled_circuit=circuit_file
            )
            print(f"Circuit for Block {i} compiled and saved to {circuit_file}.")
        except Exception as e:
            print(f"Failed to compile circuit for Block {i}, Error: {e}")

# Compile circuits
compile_all_circuits(
    blocks_path=blocks_path,
    settings_path=settings_path,
    compiled_model_path=compiled_model_path
)


In [ ]:
# Get SRS

def get_srs_for_all_blocks(settings_path, srs_path):
    """
    Generates the SRS for all blocks based on their settings.

    Args:
    - settings_path (str): Path to the directory containing settings files.
    - srs_path (str): Path to save the SRS file.

    Returns:
    - None
    """
    try:
        # Assuming all blocks share the same SRS
        print(f"Generating SRS using the settings of a block...")
        first_settings_file = os.path.join(settings_path, "block_10settings.json")

        # Generate the SRS
        ezkl.get_srs(
            srs_path=srs_path,
            settings_path=first_settings_file
        )
        print(f"SRS generated and saved to {srs_path}.")

    except Exception as e:
        print(f"Failed to generate SRS, Error: {e}")


# Generate SRS
get_srs_for_all_blocks(
    settings_path=settings_path,
    srs_path=srs_path
)


In [ ]:
# Runs the setup process

def setup_all_blocks(compiled_model_path, srs_path, vk_path, pk_path):
    """
    Run `ezkl.setup` for all blocks to generate verification and proving keys.

    Args:
    - compiled_model_path (str): Path to the directory containing compiled model files.
    - srs_path (str): Path to the structured reference string (SRS) file.
    - vk_path (str): Path to the directory to save verification keys.
    - pk_path (str): Path to the directory to save proving keys.

    Returns:
    - None
    """
    for i in range(1, 17):  # Loop through all 16 blocks
        try:
            compiled_model_file = os.path.join(compiled_model_path, f"block_{i}.compiled")
            vk_file = os.path.join(vk_path, f"block_{i}_key.vk")
            pk_file = os.path.join(pk_path, f"block_{i}_key.pk")

            print(f"Setting up for Block {i}...")

            # Run ezkl.setup
            ezkl.setup(
                model=compiled_model_file,
                vk_path=vk_file,
                pk_path=pk_file,
                srs_path=srs_path
            )

            print(f"Setup completed for Block {i}. Verification key saved to {vk_file}, Proving key saved to {pk_file}.")

        except Exception as e:
            print(f"Failed to set up for Block {i}, Error: {e}")


# Run setup for all blocks
setup_all_blocks(compiled_model_path, srs_path, vk_path, pk_path)


Setting up for Block 1...
Setup completed for Block 1. Verification key saved to /home/pxk409/Skin_EZKL/vk/block_1_key.vk, Proving key saved to /scratch/pioneer/jobs/pxk409/zk_setup/pk/block_1_key.pk.
Setting up for Block 2...
Setup completed for Block 2. Verification key saved to /home/pxk409/Skin_EZKL/vk/block_2_key.vk, Proving key saved to /scratch/pioneer/jobs/pxk409/zk_setup/pk/block_2_key.pk.
Setting up for Block 3...
Setup completed for Block 3. Verification key saved to /home/pxk409/Skin_EZKL/vk/block_3_key.vk, Proving key saved to /scratch/pioneer/jobs/pxk409/zk_setup/pk/block_3_key.pk.
Setting up for Block 4...
Setup completed for Block 4. Verification key saved to /home/pxk409/Skin_EZKL/vk/block_4_key.vk, Proving key saved to /scratch/pioneer/jobs/pxk409/zk_setup/pk/block_4_key.pk.
Setting up for Block 5...
Setup completed for Block 5. Verification key saved to /home/pxk409/Skin_EZKL/vk/block_5_key.vk, Proving key saved to /scratch/pioneer/jobs/pxk409/zk_setup/pk/block_5_key

In [ ]:
# Generate Witnesses: Runs the forward pass operation to generate a witness


import asyncio

async def generate_witnesses_for_all_blocks(calibration_path, compiled_model_path, witness_path, vk_path, srs_path):
    """
    Generates witnesses for all blocks.

    Args:
    - blocks_path (str): Path to the directory containing the ONNX blocks.
    - calibration_path (str): Path to the directory containing calibration data for each block.
    - compiled_model_path (str): Path to the directory containing compiled models for each block.
    - witness_path (str): Path to save the witness files for each block.
    - vk_path (str): Path to save the verification key for each block.
    - srs_path (str): Path to the SRS file.

    Returns:
    - None
    """
    for i in range(1, 17):  # Loop through all blocks
        try:
            print(f"Generating witness for Block {i}...")

            # Define paths for the current block
            calibration_file = os.path.join(calibration_path, f"block_{i}_calibration.json")
            compiled_model_file = os.path.join(compiled_model_path, f"block_{i}.compiled")
            witness_file = os.path.join(witness_path, f"block_{i}_witness.json")
            vk_file = os.path.join(vk_path, f"block_{i}_key.vk")

            # Generate the witness
            await ezkl.gen_witness(
                data=calibration_file,
                model=compiled_model_file,
                output=witness_file,
                vk_path=vk_file,
                srs_path=srs_path
            )

            print(f"Witness for Block {i} generated and saved to {witness_file}.")

        except Exception as e:
            print(f"Failed to generate witness for Block {i}, Error: {e}")


# Generate witnesses for all blocks
await generate_witnesses_for_all_blocks(
    calibration_path=calibration_path,
    compiled_model_path=compiled_model_path,
    witness_path=witness_path,
    vk_path=vk_path,
    srs_path=srs_path
)


Generating witness for Block 1...
Witness for Block 1 generated and saved to /home/pxk409/Skin_EZKL/witness/block_1_witness.json.
Generating witness for Block 2...
Witness for Block 2 generated and saved to /home/pxk409/Skin_EZKL/witness/block_2_witness.json.
Generating witness for Block 3...
Witness for Block 3 generated and saved to /home/pxk409/Skin_EZKL/witness/block_3_witness.json.
Generating witness for Block 4...
Witness for Block 4 generated and saved to /home/pxk409/Skin_EZKL/witness/block_4_witness.json.
Generating witness for Block 5...
Witness for Block 5 generated and saved to /home/pxk409/Skin_EZKL/witness/block_5_witness.json.
Generating witness for Block 6...
Witness for Block 6 generated and saved to /home/pxk409/Skin_EZKL/witness/block_6_witness.json.
Generating witness for Block 7...
Witness for Block 7 generated and saved to /home/pxk409/Skin_EZKL/witness/block_7_witness.json.
Generating witness for Block 8...
Witness for Block 8 generated and saved to /home/pxk409/

In [ ]:
i = 1
compiled_file = os.path.join(compiled_model_path, f"block_{i}.compiled")
witness_file  = os.path.join(witness_path,        f"block_{i}_witness.json")
pk_file       = os.path.join(pk_path,             f"block_{i}_key.pk")
proof_file    = os.path.join(proof_path,          f"block_{i}_proof.json")

ezkl.prove(
    witness    = witness_file,
    model      = compiled_file,
    pk_path    = pk_file,
    proof_path = proof_file,
    srs_path   = srs_path,
    proof_type = "for-aggr",
)

In [ ]:
i = 2
compiled_file = os.path.join(compiled_model_path, f"block_{i}.compiled")
witness_file  = os.path.join(witness_path,        f"block_{i}_witness.json")
pk_file       = os.path.join(pk_path,             f"block_{i}_key.pk")
proof_file    = os.path.join(proof_path,          f"block_{i}_proof.json")

ezkl.prove(
    witness    = witness_file,
    model      = compiled_file,
    pk_path    = pk_file,
    proof_path = proof_file,
    srs_path   = srs_path,
    proof_type = "for-aggr",
)

In [ ]:
i = 3
compiled_file = os.path.join(compiled_model_path, f"block_{i}.compiled")
witness_file  = os.path.join(witness_path,        f"block_{i}_witness.json")
pk_file       = os.path.join(pk_path,             f"block_{i}_key.pk")
proof_file    = os.path.join(proof_path,          f"block_{i}_proof.json")

ezkl.prove(
    witness    = witness_file,
    model      = compiled_file,
    pk_path    = pk_file,
    proof_path = proof_file,
    srs_path   = srs_path,
    proof_type = "for-aggr",
)

In [ ]:
i = 4
compiled_file = os.path.join(compiled_model_path, f"block_{i}.compiled")
witness_file  = os.path.join(witness_path,        f"block_{i}_witness.json")
pk_file       = os.path.join(pk_path,             f"block_{i}_key.pk")
proof_file    = os.path.join(proof_path,          f"block_{i}_proof.json")

ezkl.prove(
    witness    = witness_file,
    model      = compiled_file,
    pk_path    = pk_file,
    proof_path = proof_file,
    srs_path   = srs_path,
    proof_type = "for-aggr",
)

In [ ]:
i = 5
compiled_file = os.path.join(compiled_model_path, f"block_{i}.compiled")
witness_file  = os.path.join(witness_path,        f"block_{i}_witness.json")
pk_file       = os.path.join(pk_path,             f"block_{i}_key.pk")
proof_file    = os.path.join(proof_path,          f"block_{i}_proof.json")

ezkl.prove(
    witness    = witness_file,
    model      = compiled_file,
    pk_path    = pk_file,
    proof_path = proof_file,
    srs_path   = srs_path,
    proof_type = "for-aggr",
)

In [ ]:
i = 6
compiled_file = os.path.join(compiled_model_path, f"block_{i}.compiled")
witness_file  = os.path.join(witness_path,        f"block_{i}_witness.json")
pk_file       = os.path.join(pk_path,             f"block_{i}_key.pk")
proof_file    = os.path.join(proof_path,          f"block_{i}_proof.json")

ezkl.prove(
    witness    = witness_file,
    model      = compiled_file,
    pk_path    = pk_file,
    proof_path = proof_file,
    srs_path   = srs_path,
    proof_type = "for-aggr",
)

In [ ]:
i = 7
compiled_file = os.path.join(compiled_model_path, f"block_{i}.compiled")
witness_file  = os.path.join(witness_path,        f"block_{i}_witness.json")
pk_file       = os.path.join(pk_path,             f"block_{i}_key.pk")
proof_file    = os.path.join(proof_path,          f"block_{i}_proof.json")

ezkl.prove(
    witness    = witness_file,
    model      = compiled_file,
    pk_path    = pk_file,
    proof_path = proof_file,
    srs_path   = srs_path,
    proof_type = "for-aggr",
)

In [ ]:
i = 8
compiled_file = os.path.join(compiled_model_path, f"block_{i}.compiled")
witness_file  = os.path.join(witness_path,        f"block_{i}_witness.json")
pk_file       = os.path.join(pk_path,             f"block_{i}_key.pk")
proof_file    = os.path.join(proof_path,          f"block_{i}_proof.json")

ezkl.prove(
    witness    = witness_file,
    model      = compiled_file,
    pk_path    = pk_file,
    proof_path = proof_file,
    srs_path   = srs_path,
    proof_type = "for-aggr",
)

In [ ]:
i = 9
compiled_file = os.path.join(compiled_model_path, f"block_{i}.compiled")
witness_file  = os.path.join(witness_path,        f"block_{i}_witness.json")
pk_file       = os.path.join(pk_path,             f"block_{i}_key.pk")
proof_file    = os.path.join(proof_path,          f"block_{i}_proof.json")

ezkl.prove(
    witness    = witness_file,
    model      = compiled_file,
    pk_path    = pk_file,
    proof_path = proof_file,
    srs_path   = srs_path,
    proof_type = "for-aggr",
)

In [ ]:
i = 10
compiled_file = os.path.join(compiled_model_path, f"block_{i}.compiled")
witness_file  = os.path.join(witness_path,        f"block_{i}_witness.json")
pk_file       = os.path.join(pk_path,             f"block_{i}_key.pk")
proof_file    = os.path.join(proof_path,          f"block_{i}_proof.json")

ezkl.prove(
    witness    = witness_file,
    model      = compiled_file,
    pk_path    = pk_file,
    proof_path = proof_file,
    srs_path   = srs_path,
    proof_type = "for-aggr",
)

In [ ]:
i = 11
compiled_file = os.path.join(compiled_model_path, f"block_{i}.compiled")
witness_file  = os.path.join(witness_path,        f"block_{i}_witness.json")
pk_file       = os.path.join(pk_path,             f"block_{i}_key.pk")
proof_file    = os.path.join(proof_path,          f"block_{i}_proof.json")

ezkl.prove(
    witness    = witness_file,
    model      = compiled_file,
    pk_path    = pk_file,
    proof_path = proof_file,
    srs_path   = srs_path,
    proof_type = "for-aggr",
)

In [ ]:
i = 12
compiled_file = os.path.join(compiled_model_path, f"block_{i}.compiled")
witness_file  = os.path.join(witness_path,        f"block_{i}_witness.json")
pk_file       = os.path.join(pk_path,             f"block_{i}_key.pk")
proof_file    = os.path.join(proof_path,          f"block_{i}_proof.json")

ezkl.prove(
    witness    = witness_file,
    model      = compiled_file,
    pk_path    = pk_file,
    proof_path = proof_file,
    srs_path   = srs_path,
    proof_type = "for-aggr",
)

In [ ]:
i = 13
compiled_file = os.path.join(compiled_model_path, f"block_{i}.compiled")
witness_file  = os.path.join(witness_path,        f"block_{i}_witness.json")
pk_file       = os.path.join(pk_path,             f"block_{i}_key.pk")
proof_file    = os.path.join(proof_path,          f"block_{i}_proof.json")

ezkl.prove(
    witness    = witness_file,
    model      = compiled_file,
    pk_path    = pk_file,
    proof_path = proof_file,
    srs_path   = srs_path,
    proof_type = "for-aggr",
)

In [ ]:
i = 14
compiled_file = os.path.join(compiled_model_path, f"block_{i}.compiled")
witness_file  = os.path.join(witness_path,        f"block_{i}_witness.json")
pk_file       = os.path.join(pk_path,             f"block_{i}_key.pk")
proof_file    = os.path.join(proof_path,          f"block_{i}_proof.json")

ezkl.prove(
    witness    = witness_file,
    model      = compiled_file,
    pk_path    = pk_file,
    proof_path = proof_file,
    srs_path   = srs_path,
    proof_type = "for-aggr",
)

In [ ]:
i = 15
compiled_file = os.path.join(compiled_model_path, f"block_{i}.compiled")
witness_file  = os.path.join(witness_path,        f"block_{i}_witness.json")
pk_file       = os.path.join(pk_path,             f"block_{i}_key.pk")
proof_file    = os.path.join(proof_path,          f"block_{i}_proof.json")

ezkl.prove(
    witness    = witness_file,
    model      = compiled_file,
    pk_path    = pk_file,
    proof_path = proof_file,
    srs_path   = srs_path,
    proof_type = "for-aggr",
)

In [ ]:
i = 16
compiled_file = os.path.join(compiled_model_path, f"block_{i}.compiled")
witness_file  = os.path.join(witness_path,        f"block_{i}_witness.json")
pk_file       = os.path.join(pk_path,             f"block_{i}_key.pk")
proof_file    = os.path.join(proof_path,          f"block_{i}_proof.json")

ezkl.prove(
    witness    = witness_file,
    model      = compiled_file,
    pk_path    = pk_file,
    proof_path = proof_file,
    srs_path   = srs_path,
    proof_type = "for-aggr",
)

In [ ]:
agg_srs_path = os.path.join(base_path, "aggregate", "kzg_agg28.srs")

ezkl.get_srs(
    settings_path=None,
    logrows=28,
    srs_path=agg_srs_path,
    commitment=ezkl.PyCommitments.KZG
)

<Future pending cb=[<builtins.PyDoneCallback object at 0x7fe1f7136470>()]>

In [ ]:
# Setup aggregate circuit KZG
import glob, ezkl, os, math

proof_dir   = "/home/pxk409/Skin_EZKL/proof"
proof_files = sorted(glob.glob(os.path.join(proof_dir, "*_proof.json")))

aggregate_path  = os.path.join(base_path, "aggregate")
agg_pk_path     = os.path.join(aggregate_path, "agg.pk")
agg_vk_path     = os.path.join(aggregate_path, "agg.vk")

print("Setting up aggregation circuit …", flush=True)

ezkl.setup_aggregate(
    sample_snarks=['/home/pxk409/Skin_EZKL/proof/block_13_proof.json','/home/pxk409/Skin_EZKL/proof/block_14_proof.json'],
    vk_path=agg_vk_path,
    pk_path=agg_pk_path,
    split_proofs=True,
    logrows=26,
    srs_path="/home/pxk409/Skin_EZKL/aggregate/kzg_agg28.srs"
)

print("✅ Aggregation setup complete.", flush=True)

In [ ]:
# Setup aggregate circuit KZG
import glob, ezkl, os, math

proof_dir   = "/home/pxk409/Skin_EZKL/proof"
proof_files = sorted(glob.glob(os.path.join(proof_dir, "*_proof.json")))

aggregate_path  = os.path.join(base_path, "aggregate")
agg_pk_path     = os.path.join(aggregate_path, "agg.pk")
agg_vk_path     = os.path.join(aggregate_path, "agg.vk")

print("Setting up aggregation circuit …", flush=True)

ezkl.setup_aggregate(
    sample_snarks=['/home/pxk409/Skin_EZKL/proof/block_1_proof.json','/home/pxk409/Skin_EZKL/proof/block_2_proof.json'],
    vk_path=agg_vk_path,
    pk_path=agg_pk_path,
    split_proofs=True,
    logrows=26,
    srs_path="/home/pxk409/Skin_EZKL/aggregate/kzg_agg28.srs"
)

print("✅ Aggregation setup complete.", flush=True)

In [ ]:
# Prove aggregate
ezkl.aggregate(
    proof_files,            # list of inner proofs
    agg_proof_path,         # output aggregated proof
    agg_pk_path,            # proving key
    "evm",                  # commitment scheme ("evm" or "kzg")
    agg_logrows,
    "safe"                  # check_mode
)
assert os.path.isfile(agg_proof_path)

In [ ]:
# Verify aggregated proof
assert ezkl.verify_aggr(agg_proof_path, agg_vk_path, agg_logrows) is True
print(f"Aggregated proof verified  (wall {tv1-tv0:.2f}s, CPU {tc1-tc0:.2f}s)")

In [ ]:
# VERIFY IT
res = ezkl.verify(
        proof_path="/home/pxk409/Skin_EZKL/proof/block_5_proof.json",
        settings_path="/home/pxk409/Skin_EZKL/settings/block_5_settings.json",
        vk_path="/home/pxk409/Skin_EZKL/vk/block_5_key.vk",
        srs_path="/home/pxk409/Skin_EZKL/kzg.srs",
    )

assert res == True
print("verified", flush=True)

verified


In [ ]:
# VERIFY IT
res = ezkl.verify(
        proof_path="/home/pxk409/Skin_EZKL/proof/block_8_proof.json",
        settings_path="/home/pxk409/Skin_EZKL/settings/block_8_settings.json",
        vk_path="/home/pxk409/Skin_EZKL/vk/block_8_key.vk",
        srs_path="/home/pxk409/Skin_EZKL/kzg.srs",
    )

assert res == True
print("verified", flush=True)

verified
